# Phase 5 — validation threshold analysis

Thresholds are selected on validation only. The test set remains untouched.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "processed" / "train.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.evaluation.pipeline import load_splits, build_model_specs, classification_metrics, select_threshold
splits = load_splits(ROOT)
X_train, y_train = splits["train"]
X_validation, y_validation = splits["validation"]
X_test, y_test = splits["test"]
models = build_model_specs(X_train, y_train)
rows = []
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    scores = estimator.predict_proba(X_validation)[:, 1]
    selection = select_threshold(y_validation, scores, objective="f1")
    rows.append({"model": name, "selected_threshold": selection.threshold, "validation_f1": selection.validation_f1, "validation_balanced_accuracy": selection.validation_balanced_accuracy})
pd.DataFrame(rows).sort_values("validation_f1", ascending=False)